In [ ]:
import os
import scanpy as sc
import scanpy.external as sce
import pandas as pd
import warnings

# Suppress pandas warnings for a cleaner output
warnings.filterwarnings('ignore')

# Configure Scanpy settings for high-quality figure exports
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=150, facecolor='white', figsize=(5, 5))
sc.settings.figdir = './results/figures/' # Directory for saving output figures
os.makedirs(sc.settings.figdir, exist_ok=True)

def perform_qc_and_filtering(adata, min_genes=200, max_genes=5500, max_mt=15, min_cells=3, min_sample_size=30):
    """
    Perform Quality Control (QC), doublet removal, and basic filtering on the AnnData object.
    """
    print(f"[*] Initial dimensions: {adata.n_obs} cells, {adata.n_vars} genes.")

    # 1. Mitochondrial QC
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

    # Generate and save pre-filtering QC violin plots
    sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
                 jitter=0.4, multi_panel=True, show=False, save='_pre_filter_QC.pdf')

    # 2. Hard Filtering (Cells and Genes)
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)
    adata = adata[adata.obs.pct_counts_mt < max_mt, :].copy()
    adata = adata[adata.obs.n_genes < max_genes, :].copy()

    # 3. Filter out samples with insufficient cell counts (< 30 cells)
    sample_counts = adata.obs["Sample"].value_counts()
    removed_samples = sample_counts[sample_counts < min_sample_size]
    valid_samples = sample_counts[sample_counts >= min_sample_size].index

    if len(removed_samples) > 0:
        print(f"[-] Removed {len(removed_samples)} samples with < {min_sample_size} cells: {list(removed_samples.index)}")
    adata = adata[adata.obs["Sample"].isin(valid_samples)].copy()

    # 4. Remove Doublets using Scrublet
    print("[*] Running Scrublet for doublet detection...")
    sce.pp.scrublet(adata, batch_key="Sample", verbose=False)
    doublets = adata.obs['predicted_doublet'].sum()
    print(f"[-] Removed {doublets} predicted doublets.")
    adata = adata[~adata.obs['predicted_doublet']].copy()

    print(f"[*] Cells remaining after QC pipeline: {adata.n_obs}")
    return adata


def normalize_and_integrate(adata, batch_key="Sample"):
    """
    Normalize data, identify highly variable genes (HVGs), run PCA,
    and perform batch-effect integration using Harmony.
    """
    # Preserve raw counts before normalization
    adata.layers["counts"] = adata.X.copy()

    print("[*] Normalizing data and identifying highly variable genes...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, flavor='seurat', n_top_genes=2000, batch_key=batch_key)

    print("[*] Running PCA and Harmony integration...")
    sc.tl.pca(adata, svd_solver='arpack', use_highly_variable=True)

    # Export PCA variance ratio plot
    sc.pl.pca_variance_ratio(adata, log=True, n_pcs=30, show=False, save='_variance.pdf')

    # Execute Harmony integration
    sce.pp.harmony_integrate(adata, key=batch_key)

    print("[*] Constructing neighborhood graph and UMAP embeddings...")
    # NOTE: Neighborhood graph is built ONLY ONCE using the Harmony-corrected PCA space
    sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=20)
    sc.tl.umap(adata)

    # Export UMAP visualizations
    sc.pl.umap(adata, color=['Type'], show=False, save='_celltypes.pdf')
    sc.pl.umap(adata, color=[batch_key], show=False, save='_samples_integrated.pdf')

    return adata


def plot_canonical_markers(adata):
    """
    Generate a dot plot for canonical lineage markers to validate cell type annotations.
    """
    print("[*] Generating canonical marker gene dot plot...")
    marker_genes_dict = {
        'T cells': ['CD3D', 'CD3E', 'CD2', 'CD3G'],
        'B cells': ['CD79A', 'MS4A1', 'CD79B', 'FCRL5'],
        'TAMs': ['CD68', 'CD14'],
        'CAFs': ['COL3A1', 'COL1A2', 'DCN'],
        'TECs': ['VWF', 'PECAM1', 'PLVAP'],
    }

    sc.pl.dotplot(
        adata,
        marker_genes_dict,
        groupby='Type',
        standard_scale='var',
        dendrogram=True,
        dot_max=0.8,
        dot_min=0.05,
        colorbar_title='Mean expression',
        size_title='Fraction of cells',
        title='Lineage marker expression',
        cmap='Reds',
        show=False,
        save='_lineage_markers.pdf' # Automatically saves as dotplot_lineage_markers.pdf
    )

# ==========================================
# EXECUTION BLOCK
# ==========================================
# Assuming 'adata' has been loaded from the previous module
# adata = build_annotated_adata(adata)

# 1. Execute Quality Control
adata = perform_qc_and_filtering(adata)

# 2. Integrate and reduce dimensionality
adata = normalize_and_integrate(adata, batch_key="Sample")

# 3. Visualize marker genes (Outputs Figure 1B for the manuscript)
plot_canonical_markers(adata)

# 4. Save the cleaned and processed dataset
OUTPUT_PATH = "./data/GSE151530_processed.h5ad"
adata.write_h5ad(OUTPUT_PATH, compression='gzip')
print(f"[*] Processing complete! Clean dataset saved to {OUTPUT_PATH}")